# W06A B · 적합 전후 그래프와 웹 앱

2026-10-05 · **R 커널** · 런타임→모두 실행. A/B는 각각 독립 실행됩니다. 저장된 출력과 현재 런타임의 객체는 다릅니다.

**먼저 예상→실행→설명** 순서로 진행합니다. R/QMD/webR와 Colab은 대체 환경이므로 한 경로만 선택하세요. 예제 코드는 바로 실행되며 연습 셀은 주석으로 되어 있어 기본 흐름을 막지 않습니다. 연습 때 `#`를 지우고 완성하세요.

데이터 역할을 구별하세요: Anscombe/Hamilton은 설명용 구성 자료, attitude는30개 부서의 실제 설문 집계, 잔차 패턴은 조건을 정한 모의 자료입니다. 수업 질문은 `주어진 회귀식에 무엇이 남았는가?`입니다.

공통 코드 고정 버전: **2026-fall-w06a**. [함수 입력·출력·정의 안내](https://github.com/lunalab-ai/regre/blob/2026-fall-w06a/src/diagnostics-api.md). `source()`는 정의를 읽고, `lm()`은 모형을 적합하며, 진단 함수는 그 결과를 읽습니다. 캐시 재사용 때도 파일 동일성을 확인합니다. 런타임 재시작 후 준비 셀부터 다시 실행하세요.

In [ ]:
# 새 R 런타임에서도 단독 실행됩니다. API 키/수동 업로드가 필요 없습니다.
tag <- "2026-fall-w06a"
local_dirs <- c("labs/student/w06a/data","../../labs/student/w06a/data")
found <- local_dirs[file.exists(file.path(local_dirs,"diagnostics.R"))]
data_dir <- if(length(found)) found[1] else file.path(path.expand("~"),".cache","regre",tag)
dir.create(data_dir,recursive=TRUE,showWarnings=FALSE)
expected <- c("diagnostics.R"="1d6d198179eb52922e0b7da3a3836838","hamilton.csv"="9dfd652f3a9efb8d751961d609bca459")
paths <- c("diagnostics.R"="src/diagnostics.R","hamilton.csv"="data/sample/hamilton.csv")
options(timeout=120)
for(name in names(paths)) {
  target <- file.path(data_dir,name)
  if(!file.exists(target)) {
    url <- paste0("https://raw.githubusercontent.com/lunalab-ai/regre/",tag,"/",paths[[name]])
    status <- download.file(url,target,mode="wb")
    if(status!=0) stop("다운로드 실패: 인터넷을 확인하고 준비 셀을 재실행하세요.")
    cat(name,": 고정 버전 최초 다운로드\n")
  } else cat(name,": 동봉 파일 또는 캐시 확인\n")
  if(unname(tools::md5sum(target))!=expected[[name]]) stop("파일 동일성 확인 실패. 수정 파일은 보관하고 고정 버전으로 다시 시작하세요.")
}
source(file.path(data_dir,"diagnostics.R"))
d <- datasets::attitude
fit <- lm(rating~complaints+learning,data=d)
res <- diag_residuals(fit)
ham <- diag_hamilton(file.path(data_dir,"hamilton.csv"))
cat(R.version.string,"\n30 departments; Hamilton: 15 constructed observations\n")

## 5. 적합 전: 쌍별 그림과 함께 보는 모형

[diag_hamilton 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w06a/src/diagnostics.R#L36)으로 읽은 Hamilton은 교재 표5.1의15개 구성 관측입니다. [diag_hamilton_fits 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w06a/src/diagnostics.R#L45)는 같은 Y/행의 세 모형 R²·SSE·rank를 나란히 반환합니다.

In [ ]:
ham
pairs(ham,pch=19,col="#087f8c",main="Hamilton: inspect all pairs")
ham_summary <- diag_hamilton_fits(ham)
ham_summary
cor(ham)
coef(lm(Y~X1+X2,data=ham))

### L5 · 예측–실행–설명

Y와X1의 쌍별 상관이 거의0이므로 X1을 무조건 제외해도 될까요? 한 변수/두 변수 모형 결과로 설명하세요.

In [ ]:
# c(one=ham_summary$R2[...],both=ham_summary$R2[...])
# 비교 조건과 삭제할 수 없는 이유를 적으세요.

## 6. 회전은 자료를 바꾸는가?

[diag_project 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w06a/src/diagnostics.R#L53)는 X1/X2/Y를 각각 표준화한 뒤 직교투영합니다. azimuth=35°, elevation=20°가 기본입니다. 반환 u/v/depth는 화면 좌표이며 원래 단위가 아닙니다. 데이터와 적합계수는 바뀌지 않습니다.

In [ ]:
projection <- diag_project(ham,azimuth=35,elevation=20)
selected <- 4L
plot(projection$u,projection$v,pch=19,col="#087f8c",xlab="Screen u (scaled)",ylab="Screen v (scaled)",asp=1)
points(projection$u[selected],projection$v[selected],col="#d95d52",pch=19,cex=1.6)
text(projection$u[selected],projection$v[selected],labels=selected,pos=3)
ham[selected,]

### L6 · 예측–실행–설명

방위각을125°로 바꾸고4번점을 다시 찾으세요. 바뀐 값과 유지되는 값을 구별하세요.

In [ ]:
# second_view <- diag_project(ham,azimuth=...,elevation=20)
# plot(second_view$u,second_view$v)

## 7. 적합 후: 관찰·가능한 설명·다음 확인

§5.6의 세 목적을 이해하는 추가 연결 예시입니다. 정규성 검정/Cook 거리의 상세 절차는 다음 범위에 남깁니다. 모의 자료의 order는 수집 순서를 뜻하지만 attitude 행번호는 시간이 아닙니다. [diag_patterns 정의](https://github.com/lunalab-ai/regre/blob/2026-fall-w06a/src/diagnostics.R#L68)

In [ ]:
patterns <- diag_patterns(n=100,seed=605)
op <- par(mfrow=c(2,2),mar=c(4,4,2,1))
for(name in names(patterns)) {
  a <- patterns[[name]]; xx <- if(name=="order") a$order else a$fitted
  plot(xx,a$residual,pch=19,col="#087f8c",main=paste("Simulation:",name),
       xlab=if(name=="order") "Collection order" else "Fitted",ylab="Residual")
  abline(h=0,lty=2)
}
par(op)

### L7 · 예측–실행–설명

fan 그림을 관찰한 사실, 가능한 설명, 추가 확인의 세 문장으로 쓰세요. 계수에 반드시 편향이 생겼다고 말해도 될까요?

In [ ]:
# observation <- "..."
# possible <- "..."
# next_check <- "..."

## 8. 마지막: 웹 앱에서 같은 점을 연결하기

[누적 Shiny 편집기](https://lunalab-ai.github.io/regre/apps/w06a/edit/index.html)를 여세요. W06A 잔차에서 부서와 잔차종류를 바꾸고, W06A 회전에서 각도와 점을 바꿉니다. `W06A-L8`을 찾아 출력문을 수정→Run→Download 합니다. R Colab에서 별도 Shiny 서버를 띄울 필요는 없습니다.

In [ ]:
chosen_row <- res[1,]
cat("앱과 같은 관측: 부서",chosen_row$row,"\n")
# 위 편집기에서 입력→선택한 행→선택한 열→출력의 연결을 확인합니다.

### L8 · 예측–실행–설명

앱의 안내 문장을 선택된 잔차값이 소수셋째자리로 나오는 문장으로 바꾸세요. raw/internal/external 전환 시 출력값도 변해야 합니다.

In [ ]:
# 앱의 W06A-L8 renderText 안에서 고칩니다.
# sprintf("선택 잔차: %.3f", ...)
# 이 셀의 주석은 앱 코드로 자동 전송되지 않습니다.

In [ ]:
stopifnot(nrow(ham)==15,ham$X2[4]==6.64,ham_summary$rank[3]==3,abs(ham_summary$R2[3]-.999847)<1e-6,nrow(projection)==15,length(patterns)==4)